In [1]:
import os
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
import json
from typing import Any, Dict, List, Optional
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [2]:
def parse_metrics_json_dir_to_csv(
    *,
    input_dir: str,
    out_csv: str,
    method: str,
    method_label: str,
    glob_pattern: str = "**/*.json",
    keep_empty_metrics_row: bool = True,
    sort_columns: bool = True,
) -> pd.DataFrame:
    """
    Parse metrics JSONs under input_dir and save a CSV with method labels.

    Added columns:
      - method
      - method_label
    """

    def safe_get(d: Dict[str, Any], path: List[str], default=None):
        cur = d
        for k in path:
            if not isinstance(cur, dict) or k not in cur:
                return default
            cur = cur[k]
        return cur

    def flatten_dict(d: Dict[str, Any], prefix: str = "") -> Dict[str, Any]:
        out = {}
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict):
                out.update(flatten_dict(v, key))
            else:
                out[key] = v
        return out

    root = Path(input_dir).expanduser().resolve()
    files = sorted(root.glob(glob_pattern))

    rows: List[Dict[str, Any]] = []

    for fp in files:
        if not fp.is_file():
            continue

        try:
            with fp.open("r", encoding="utf-8") as f:
                obj = json.load(f)
        except Exception as e:
            print(f"[WARN] skip {fp}: {e}")
            continue

        meta = obj.get("meta", {}) if isinstance(obj, dict) else {}
        resp = obj.get("response", {}) if isinstance(obj, dict) else {}

        category = meta.get("category") or meta.get("source_category")
        window_seconds = meta.get("window_seconds")
        stride_ratio = meta.get("stride_ratio")
        video_path = meta.get("video_path").split("/")[-1]
        sample_fps = meta.get("sample_fps")
        start_frame_idx = meta.get("start_frame_idx")
        end_frame_idx = meta.get("end_frame_idx")
        num_frames = meta.get("num_frames")

        metrics_list = safe_get(obj, ["response", "metrics_list"], default=[])
        if not isinstance(metrics_list, list):
            metrics_list = []

        base = {
            "video_path": video_path,
            "category": category,
            "sample_fps": sample_fps,
            "window_seconds": window_seconds,
            "stride_ratio": stride_ratio,
            "start_frame_idx": start_frame_idx,
            "end_frame_idx": end_frame_idx,
            "num_frames": num_frames,
            "method": method,
            "method_label": method_label,
        }

        if len(metrics_list) == 0:
            if keep_empty_metrics_row:
                rows.append({**base, "metric_index": None})
            continue

        for i, m in enumerate(metrics_list):
            if not isinstance(m, dict):
                continue
            flat_m = flatten_dict(m)
            rows.append({**base, "metric_index": i, **flat_m})

    df = pd.DataFrame(rows)

    if sort_columns and not df.empty:
        base_cols = [
            "video_path",
            "category",
            "sample_fps",
            "window_seconds",
            "stride_ratio",
            "start_frame_idx",
            "end_frame_idx",
            "num_frames",
            "method",
            "method_label",
            "metric_index",
        ]
        other_cols = [c for c in df.columns if c not in base_cols]
        df = df[base_cols + sorted(other_cols)]

    Path(out_csv).expanduser().resolve().parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)

    print(
        f"[OK] dir={input_dir} | method={method} | rows={len(df)} | saved={out_csv}"
    )
    return df

In [3]:
# model_name = "InternVL3-14B"
# model_name = "Qwen3-VL-32B-Instruct"
# categorys = [
#     "abuse", "arson", "fighting", "shoplifting", "shooting",
#     "vandalism", "stealing", "arrest", "roadaccidents",
#     "burglary", "explosion", "robbery", "assault"
# ]
# root_path = "/home/users/ntu/wenyanch/scratch/github/"
# with_cacheblend_dir = (
#     f"{root_path}lmcache-multimodal/scripts_test_video/results_analysis/"
#     f"logs/{model_name}/small_dataset"
# )
# without_cacheblend_dir = (
#     f"{root_path}lmcache-multimodal/scripts_test_video/results_analysis/"
#     f"logs_baselines/{model_name}/small_dataset"
# )
# methods = ["w_cacheblend", "without_cacheblend"]
# method_labels = ["w/ CacheBlend", "w/o CacheBlend"]
# dirs = [with_cacheblend_dir, without_cacheblend_dir]
# win_sizes = [40]
# stride_sizes = [20]
# fps = 2.0
# dfs = []


# for win_size in win_sizes:
#     for stride_size in stride_sizes:
#         for dir_path, method, method_label in zip(dirs, methods, method_labels):
#             input_dir = (
#                 f"{dir_path}/anomaly_win{win_size}s_"
#                 f"stride{stride_size}pct_fps{fps}"
#             )

#             df = parse_metrics_json_dir_to_csv(
#                 input_dir=input_dir,
#                 out_csv=f"{input_dir}/parsed_metrics.csv",
#                 method=method,
#                 method_label=method_label,
#             )
#             dfs.append(df)

#         final_df = pd.concat(dfs, ignore_index=True)
#         final_df.to_csv(f"{root_path}/lmcache-multimodal/scripts_test_video/results_analysis/csv/{model_name}/small_dataset/win{win_size}_stride{stride_size}_latancy_results.csv", index=False)

#         print(f"[OK] final rows = {len(final_df)}")

In [4]:
def parse_all_metrics_json_dir_to_csv(
    recompute_ratio,
    input_dir: str,
    out_csv: str,
    method: str,
    method_label: str,
    glob_pattern: str = "**/*.json",
    keep_empty_metrics_row: bool = True,
    sort_columns: bool = True,
) -> pd.DataFrame:
    """
    Parse metrics JSONs under input_dir and save a CSV with method labels.

    Added columns:
      - method
      - method_label
    """

    def safe_get(d: Dict[str, Any], path: List[str], default=None):
        cur = d
        for k in path:
            if not isinstance(cur, dict) or k not in cur:
                return default
            cur = cur[k]
        return cur

    def flatten_dict(d: Dict[str, Any], prefix: str = "") -> Dict[str, Any]:
        out = {}
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict):
                out.update(flatten_dict(v, key))
            else:
                out[key] = v
        return out

    root = Path(input_dir).expanduser().resolve()
    print(f'recompute_ratio is {recompute_ratio}, root is {root}')
    files = sorted(root.glob(glob_pattern))

    rows: List[Dict[str, Any]] = []

    for fp in files:
        if not fp.is_file():
            continue

        try:
            with fp.open("r", encoding="utf-8") as f:
                obj = json.load(f)
        except Exception as e:
            print(f"[WARN] skip {fp}: {e}")
            continue

        meta = obj.get("meta", {}) if isinstance(obj, dict) else {}
        resp = obj.get("response", {}) if isinstance(obj, dict) else {}
        choices = resp.get("choices", [])
        if not choices:
            continue
        msg = choices[0].get("message", {})
        content = str(msg.get("content", "")).strip()
        # Parse label from content, expected "Yes" or "No"
        if "Yes" in content:
            est = "Yes"
        elif "No" in content:
            est = "No"
        else:
            continue  # skip invalid label

        category = meta.get("category") or meta.get("source_category")
        label = meta.get("label")
        window_seconds = meta.get("window_seconds")
        stride_ratio = meta.get("stride_ratio")
        video_path = meta.get("video_path").split("/")[-1]
        sample_fps = meta.get("sample_fps")
        start_frame_idx = meta.get("start_frame_idx")
        end_frame_idx = meta.get("end_frame_idx")
        num_frames = meta.get("num_frames")


        metrics_list = safe_get(obj, ["response", "metrics_list"], default=[])
        if not isinstance(metrics_list, list):
            metrics_list = []

        base = {
            "video_path": video_path,
            "category": category,
            "sample_fps": sample_fps,
            "window_seconds": window_seconds,
            "stride_ratio": stride_ratio,
            "start_frame_idx": start_frame_idx,
            "end_frame_idx": end_frame_idx,
            "num_frames": num_frames,
            "recompute_ratio": recompute_ratio,
            "method": method,
            "method_label": method_label,
            "category": category,
            "label": label,
            "est": est
        }

        if len(metrics_list) == 0:
            if keep_empty_metrics_row:
                rows.append({**base, "metric_index": None})
            continue

        for i, m in enumerate(metrics_list):
            if not isinstance(m, dict):
                continue
            flat_m = flatten_dict(m)
            rows.append({**base, "metric_index": i, **flat_m})

    df = pd.DataFrame(rows)

    if sort_columns and not df.empty:
        base_cols = [
            "video_path",
            "category",
            "sample_fps",
            "window_seconds",
            "stride_ratio",
            "start_frame_idx",
            "end_frame_idx",
            "num_frames",
            "recompute_ratio",
            "method",
            "method_label",
            "metric_index",
        ]
        other_cols = [c for c in df.columns if c not in base_cols]
        df = df[base_cols + sorted(other_cols)]

    Path(out_csv).expanduser().resolve().parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)

    print(
        f"[OK] dir={input_dir} | method={method} | rows={len(df)} | saved={out_csv}"
    )
    return df

In [5]:
model_name = "InternVL3-14B"
model_name = "Qwen3-VL-32B-Instruct"
categorys = [
    "abuse", "arson", "fighting", "shoplifting", "shooting",
    "vandalism", "stealing", "arrest", "roadaccidents",
    "burglary", "explosion", "robbery", "assault"
]
root_path = "/home/users/ntu/yulin001/scratch/wychen/github/"
with_cacheblend_dir = (
    f"{root_path}lmcache-multimodal/scripts_test_video/results_analysis/"
    f"logs/{model_name}/small_dataset/use_gpu"
)
without_cacheblend_dir = (
    f"{root_path}lmcache-multimodal/scripts_test_video/results_analysis/"
    f"logs_baselines/{model_name}/small_dataset"
)
methods = ["w_cacheblend"]
method_labels = ["w/ CacheBlend"]
dirs = [with_cacheblend_dir]
win_sizes = [40]
stride_sizes = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
fps = 2.0
dfs = []
recompute_ratio = 0.15

for win_size in win_sizes:
    for stride_size in stride_sizes:
        for dir_path, method, method_label in zip(dirs, methods, method_labels):
            input_dir = (
                f"{dir_path}/anomaly_win{win_size}s_"
                f"stride{stride_size}pct_fps{fps}"
            )

            df = parse_all_metrics_json_dir_to_csv(
                recompute_ratio,
                input_dir=input_dir,
                out_csv=f"{input_dir}/parsed_metrics.csv",
                method=method,
                method_label=method_label,
            )
            dfs.append(df)

    final_df = pd.concat(dfs, ignore_index=True)
    final_df.to_csv(f"{root_path}/lmcache-multimodal/scripts_test_video/results_analysis/csv/{model_name}/small_dataset/use_gpu/win{win_size}_stride10-100_all_metrics_results.csv", index=False)
    print(f"[OK] final rows = {len(final_df)}")

recompute_ratio is 0.15, root is /scratch/users/ntu/yulin001/wychen/github/lmcache-multimodal/scripts_test_video/results_analysis/logs/Qwen3-VL-32B-Instruct/small_dataset/use_gpu/anomaly_win40s_stride10pct_fps2.0
[OK] dir=/home/users/ntu/yulin001/scratch/wychen/github/lmcache-multimodal/scripts_test_video/results_analysis/logs/Qwen3-VL-32B-Instruct/small_dataset/use_gpu/anomaly_win40s_stride10pct_fps2.0 | method=w_cacheblend | rows=22 | saved=/home/users/ntu/yulin001/scratch/wychen/github/lmcache-multimodal/scripts_test_video/results_analysis/logs/Qwen3-VL-32B-Instruct/small_dataset/use_gpu/anomaly_win40s_stride10pct_fps2.0/parsed_metrics.csv
recompute_ratio is 0.15, root is /scratch/users/ntu/yulin001/wychen/github/lmcache-multimodal/scripts_test_video/results_analysis/logs/Qwen3-VL-32B-Instruct/small_dataset/use_gpu/anomaly_win40s_stride20pct_fps2.0
[OK] dir=/home/users/ntu/yulin001/scratch/wychen/github/lmcache-multimodal/scripts_test_video/results_analysis/logs/Qwen3-VL-32B-Instruc

In [6]:
model_name = "InternVL3-14B"
categorys = [
    "abuse", "arson", "fighting"
]
root_path = "/home/users/ntu/yulin001/scratch/wychen/github/"
with_cacheblend_dir = (
    f"{root_path}lmcache-multimodal/scripts_test_video/results_analysis/"
    f"logs/{model_name}/reuse_tokens"
)
win_sizes = [40]
stride_sizes = [20]
fps = 2.0
recompute_rations = [0.1,0.2,0.3,0.4,0.5,0.6,0.7]
dfs = []


# for win_size in win_sizes:
#     for stride_size in stride_sizes:
#         for recompute_ratio in recompute_rations:
#             input_dir = (
#                 f"{with_cacheblend_dir}/recompute_ratio_{recompute_ratio}/anomaly_win{win_size}s_"
#                 f"stride{stride_size}pct_fps{fps}"
#             )
#             method, method_label = 'with_cacheblend', 'w/ CacheBlend'
#             df = parse_all_metrics_json_dir_to_csv(
#                 recompute_ratio,
#                 input_dir=input_dir,
#                 out_csv=f"{input_dir}/parsed_all_metrics.csv",
#                 method=method,
#                 method_label=method_label,
#             )
#             dfs.append(df)

#     final_df = pd.concat(dfs, ignore_index=True)
#     final_df.to_csv(f"{root_path}/lmcache-multimodal/scripts_test_video/results_analysis/csv/{model_name}/small_dataset/win{win_size}_stride{stride_size}_reuse_tokens_all_metrics_results.csv", index=False)
#     print(f"[OK] final rows = {len(final_df)}")